In [7]:
import tensorflow as tf
import numpy as np
import os
from PIL import Image
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

MODEL_PATH = "/content/drive/MyDrive/sort-it-right.h5"
DATASET_PATH = r"/content/drive/MyDrive/Test"
IMG_SIZE = (224, 224)

labels = ['Biodegradable', 'Non-biodegradable', 'Recyclable']

# Attempt to load the model using the legacy tf.keras to handle potential version incompatibilities
try:
    model = tf.keras.models.load_model(MODEL_PATH)
except (ValueError, TypeError) as e: # Catch both ValueError and TypeError for compatibility issues
    if "Unrecognized keyword arguments passed to DepthwiseConv2D: {'groups': 1}" in str(e):
        print("Detected Keras version incompatibility. Attempting to load with legacy Keras.")
        # Use legacy Keras API for compatibility
        import tf_keras

        # Define a custom DepthwiseConv2D class that ignores the 'groups' argument.
        # This is necessary because the saved model's configuration for DepthwiseConv2D
        # contains 'groups: 1', which is not a recognized argument for the layer's constructor
        # in current tf_keras versions.
        class FixedDepthwiseConv2D(tf_keras.layers.DepthwiseConv2D):
            def __init__(self, *args, **kwargs):
                kwargs.pop('groups', None) # Remove the problematic 'groups' argument
                super().__init__(*args, **kwargs)

        with tf_keras.utils.custom_object_scope({'DepthwiseConv2D': FixedDepthwiseConv2D}):
            model = tf_keras.models.load_model(MODEL_PATH)
    else:
        raise e

#loading the dataset

X = []
y_true = []

for label_index, label_name in enumerate(labels):

    folder_path = os.path.join(DATASET_PATH, label_name)

    # Check if folder exists before iterating
    if not os.path.exists(folder_path):
        print(f"Warning: Folder not found: {folder_path}. Skipping.")
        continue

    for file in os.listdir(folder_path):

        file_path = os.path.join(folder_path, file)

        try:
            image = Image.open(file_path).convert("RGB")
            image = image.resize(IMG_SIZE)

            img_array = np.array(image) / 255.0

            X.append(img_array)
            y_true.append(label_index)
        except Exception as e:
            print(f"Error processing image {file_path}: {e}")
            continue

X = np.array(X)
y_true = np.array(y_true)

#predictions

predictions = model.predict(X)

y_pred = np.argmax(predictions, axis=1)

#confusion matrix

cm = confusion_matrix(y_true, y_pred)

print("\nCONFUSION MATRIX:")
print(cm)

#classification report

print("\nCLASSIFICATION REPORT:")
print(classification_report(
    y_true,
    y_pred,
    target_names=labels
))

#metrics

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average='weighted'
)

recall = recall_score(
    y_true,
    y_pred,
    average='weighted'
)

f1 = f1_score(
    y_true,
    y_pred,
    average='weighted'
)

print("\nOVERALL METRICS:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

specificity_per_class = []

for i in range(len(labels)):

    TP = cm[i, i]

    FN = np.sum(cm[i, :]) - TP

    FP = np.sum(cm[:, i]) - TP

    # Ensure TN + FP is not zero to avoid division by zero for specificity
    TN = np.sum(cm) - (TP + FP + FN)
    if (TN + FP) == 0:
        specificity = 0.0 # Or NaN, depending on desired behavior for this edge case
    else:
        specificity = TN / (TN + FP)

    specificity_per_class.append(specificity)

    print(f"Specificity ({labels[i]}): {specificity:.4f}")

avg_specificity = np.mean(specificity_per_class)

print(f"\nAverage Specificity: {avg_specificity:.4f}")

Detected Keras version incompatibility. Attempting to load with legacy Keras.


5/5 [==============================] - 5s 575ms/step

CONFUSION MATRIX:
[[35  0  0]
 [ 0 60  0]
 [ 0  0 60]]

CLASSIFICATION REPORT:
                   precision    recall  f1-score   support

    Biodegradable       1.00      1.00      1.00        35
Non-biodegradable       1.00      1.00      1.00        60
       Recyclable       1.00      1.00      1.00        60

         accuracy                           1.00       155
        macro avg       1.00      1.00      1.00       155
     weighted avg       1.00      1.00      1.00       155


OVERALL METRICS:
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1 Score:  1.0000
Specificity (Biodegradable): 1.0000
Specificity (Non-biodegradable): 1.0000
Specificity (Recyclable): 1.0000

Average Specificity: 1.0000
